# 05 — Evaluation & Phase 1 vs Phase 2 Comparison

## What this notebook answers
1. Did NOAA features improve overall weighted F1?
2. Did Major (Tier 2) recall improve specifically?
3. Which NOAA features drive predictions (SHAP)?
4. Does the under-reservation exposure decrease?
5. Is the improvement consistent across matched vs unmatched disasters?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, warnings, os
warnings.filterwarnings('ignore')

from sklearn.metrics import (confusion_matrix, classification_report,
                              f1_score, recall_score, precision_score)

PROCESSED = '../data/processed/'
os.makedirs(PROCESSED + 'figures/', exist_ok=True)

with open(PROCESSED + 'best_model_v2.pkl', 'rb') as f:
    bundle = pickle.load(f)

y_test  = bundle['y_test']
preds   = bundle['preds']
proba   = bundle['proba']
names   = bundle['target_names']

print(f'Model      : {bundle["model_name"]}')
print(f'Val  F1    : {bundle["val_f1"]:.4f}')
print(f'Test F1    : {bundle["test_f1"]:.4f}')
print(f'Phase 1 F1 : {bundle["phase1_test_f1"]:.4f}')
print(f'Improvement: {bundle["test_f1"] - bundle["phase1_test_f1"]:+.4f}')

## Per-Class Metric Comparison: Phase 1 vs Phase 2
We compare recall and F1 for each funding tier to measure where the NOAA features
had the most impact. The primary target was Major (Tier 2) recall.

In [ ]:
# Phase 1 vs Phase 2 per-class comparison
from sklearn.metrics import classification_report

P1_METRICS = {
    'Minor (<$1M)':          {'precision': 0.44, 'recall': 0.64, 'f1': 0.52},
    'Moderate ($1M-$50M)':   {'precision': 0.80, 'recall': 0.76, 'f1': 0.78},
    'Major ($50M-$500M)':    {'precision': 0.57, 'recall': 0.48, 'f1': 0.52},
}

print('='*70)
print('  PHASE 1 vs PHASE 2 — Per-class metrics (test set)')
print('='*70)

report = classification_report(y_test, preds, target_names=names,
                                output_dict=True, zero_division=0)

rows = []
for tier in names:
    p2 = report[tier]
    p1 = P1_METRICS.get(tier, {})
    rows.append({
        'Tier':              tier,
        'P1 Recall':         p1.get('recall', None),
        'P2 Recall':         round(p2['recall'], 3),
        'Recall Delta':      round(p2['recall'] - p1.get('recall', p2['recall']), 3),
        'P1 F1':             p1.get('f1', None),
        'P2 F1':             round(p2['f1-score'], 3),
        'F1 Delta':          round(p2['f1-score'] - p1.get('f1', p2['f1-score']), 3),
    })

display(pd.DataFrame(rows).set_index('Tier'))
print(f'\nOverall — Phase 1 weighted F1: {bundle["phase1_test_f1"]:.4f}')
print(f'Overall — Phase 2 weighted F1: {bundle["test_f1"]:.4f}')
print(f'Overall — Improvement:         {bundle["test_f1"] - bundle["phase1_test_f1"]:+.4f}')

## Confusion Matrix
Red borders highlight cross-tier misclassifications (|true - pred| >= 2),
which represent the most costly errors operationally.

In [ ]:
cm = confusion_matrix(y_test, preds)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=names, yticklabels=names, ax=ax)

# Red border on extreme misclassifications
for i in range(len(names)):
    for j in range(len(names)):
        if abs(i - j) >= 2:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False,
                                        edgecolor='red', lw=2.5))

ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Phase 2 Confusion Matrix — {bundle["model_name"]}\n(red = cross-tier misclassification, |true−pred| >= 2)')
plt.tight_layout()
plt.savefig(PROCESSED + 'figures/confusion_matrix_v2.png', dpi=150)
plt.show()

## Cost of Misclassification — Under-Reservation Exposure
Under-reservation occurs when a disaster is predicted as a lower tier than its actual tier.
Using tier midpoints as funding proxies, we estimate the total under-reserved dollar amount
and compare it to the Phase 1 baseline.

In [ ]:
# Cost of misclassification — under-reservation exposure
TIER_MIDPOINTS = {0: 0.5e6, 1: 25.5e6, 2: 275e6}

exposure_total = 0
rows = []
for true_t in range(3):
    for pred_t in range(3):
        count = cm[true_t, pred_t]
        if count == 0 or true_t == pred_t:
            continue
        exposure_per = TIER_MIDPOINTS[pred_t] - TIER_MIDPOINTS[true_t]
        exposure_total_t = exposure_per * count
        direction = 'Under-reservation' if pred_t < true_t else 'Over-reservation'
        rows.append({
            'True Tier': names[true_t], 'Pred Tier': names[pred_t],
            'Count': count,
            'Exposure per event ($M)': round(exposure_per/1e6, 1),
            'Total exposure ($M)':     round(exposure_total_t/1e6, 1),
            'Direction': direction
        })
        if direction == 'Under-reservation':
            exposure_total += abs(exposure_total_t)

cost_df = pd.DataFrame(rows).sort_values('Total exposure ($M)')
display(cost_df)
print(f'\nTotal under-reservation exposure (Phase 2): ${exposure_total/1e6:,.1f}M')
print(f'Phase 1 baseline under-reservation:        $14,798.0M')
print(f'Reduction:                                  ${(14798 - exposure_total/1e6):,.1f}M')

## Feature Importance — NOAA vs Phase 1 Features
Red bars indicate NOAA Phase 2 features; blue bars are Phase 1 features.
This visualization shows how much the physical intensity signals contributed
relative to the contextual/demographic features from Phase 1.

In [ ]:
# Feature importance — highlight NOAA features
pipeline = bundle['pipeline']
model    = pipeline.named_steps['model']

if hasattr(model, 'feature_importances_'):
    ohe_names = pipeline.named_steps['pre'].transformers_[0][1].named_steps['ohe'].get_feature_names_out(bundle['cat_features'])
    all_names = list(ohe_names) + bundle['num_features']
    
    importances = pd.Series(model.feature_importances_, index=all_names).sort_values(ascending=False).head(20)
    
    colors = ['#e74c3c' if any(n in idx for n in ['noaa']) else '#3498db'
              for idx in importances.index]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    importances.plot(kind='barh', ax=ax, color=colors[::-1])
    ax.invert_yaxis()
    ax.set_title('Top 20 Feature Importances\n(red = NOAA Phase 2 features, blue = Phase 1 features)')
    ax.set_xlabel('Importance')
    
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='#e74c3c', label='NOAA (Phase 2)'),
                        Patch(color='#3498db', label='Phase 1 features')])
    plt.tight_layout()
    plt.savefig(PROCESSED + 'figures/feature_importance_v2.png', dpi=150)
    plt.show()
elif hasattr(model, 'coef_'):
    ohe_names = pipeline.named_steps['pre'].transformers_[0][1].named_steps['ohe'].get_feature_names_out(bundle['cat_features'])
    all_names = list(ohe_names) + bundle['num_features']
    coef_mean = np.abs(model.coef_).mean(axis=0)
    importances = pd.Series(coef_mean, index=all_names).sort_values(ascending=False).head(20)
    
    colors = ['#e74c3c' if 'noaa' in idx else '#3498db' for idx in importances.index]
    fig, ax = plt.subplots(figsize=(10, 6))
    importances.plot(kind='barh', ax=ax, color=colors[::-1])
    ax.invert_yaxis()
    ax.set_title('Top 20 Feature Coefficients (mean |coef| across classes)\n(red = NOAA Phase 2 features)')
    plt.tight_layout()
    plt.savefig(PROCESSED + 'figures/feature_coef_v2.png', dpi=150)
    plt.show()

## Critical Reflection
Summary of Phase 2 improvements, remaining limitations, and Phase 3 suggestions.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║         PHASE 2 CRITICAL REFLECTION                         ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  IMPROVEMENTS OVER PHASE 1                                   ║
║  • NOAA physical intensity features added                    ║
║  • Major recall target: 48% (P1) → see results above        ║
║  • Under-reservation exposure reduced                        ║
║                                                              ║
║  REMAINING LIMITATIONS                                       ║
║  1. NOAA match rate not 100% — biological/drought           ║
║     disasters have no NOAA event match                       ║
║  2. MAG field only populated for wind/tornado events         ║
║     (hurricanes use separate HURDAT2 track data)            ║
║  3. NOAA damage estimates are pre-FEMA and often            ║
║     undercount true public infrastructure cost              ║
║  4. Join is state-level date overlap — some false           ║
║     matches possible for states with multiple concurrent     ║
║     events in the same period                               ║
║                                                              ║
║  PHASE 3 SUGGESTIONS                                         ║
║  • Add HURDAT2 hurricane track data (storm category,        ║
║    max sustained wind, storm radius) for tropical events     ║
║  • USGS ShakeMap for earthquake intensity                    ║
║  • PRISM gridded precipitation for flood events             ║
╚══════════════════════════════════════════════════════════════╝
""")